In [1]:
import math
from dataclasses import dataclass
from typing import Optional, Dict, Any
import sys, os
sys.path.append(os.path.abspath('../src/'))
from dataset import TrainDataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()



True

In [2]:

# ---------------------------
# Utilities
# ---------------------------
class MLP(nn.Module):
    def __init__(self, d_in: int, d_hidden: int, d_out: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_out),
        )

    def forward(self, x):
        return self.net(x)


class SinCosPosEmb2D(nn.Module):
    """
    2D sin-cos positional embedding for grid (H,W).
    Returns (H*W, d_model).
    """
    def __init__(self, d_model: int, H: int, W: int):
        super().__init__()
        assert d_model % 4 == 0, "d_model must be divisible by 4 for 2D sin-cos"
        self.d = d_model
        self.H = H
        self.W = W

        pe = self._build(H, W, d_model)  # (HW, d)
        self.register_buffer("pe", pe, persistent=False)

    @staticmethod
    def _build(H, W, d):
        assert d % 4 == 0
        d_half = d // 2
        d_quarter = d // 4

        y = torch.linspace(0, 1, steps=H)
        x = torch.linspace(0, 1, steps=W)
        yy, xx = torch.meshgrid(y, x, indexing="ij")
        yy = yy.reshape(-1, 1)  # (HW,1)
        xx = xx.reshape(-1, 1)

        # ---- standard transformer div_term ----
        i = torch.arange(d_quarter, dtype=torch.float32)  # 0..d_quarter-1
        div = torch.exp(-math.log(10_000.0) * i / max(1, d_quarter - 1))  # (d_quarter,)

        y_arg = yy * div  # (HW,d_quarter)
        x_arg = xx * div

        pe = torch.cat([torch.sin(y_arg), torch.cos(y_arg),
                        torch.sin(x_arg), torch.cos(x_arg)], dim=-1)
        return pe.float()


    def forward(self):
        return self.pe  # (HW, d)


# ---------------------------
# Slot Attention (Locatello-style)
# ---------------------------
class SlotAttention(nn.Module):
    """
    Inputs:  x (B, N, D_in)
    Outputs: slots (B, K, D_slot)
    """
    def __init__(
        self,
        num_slots: int,
        dim_in: int,
        dim_slot: int,
        iters: int = 3,
        eps: float = 1e-8,
        hidden_dim: int = 128,
    ):
        super().__init__()
        self.K = num_slots
        self.iters = iters
        self.eps = eps
        self.dim_slot = dim_slot

        self.norm_in = nn.LayerNorm(dim_in)
        self.norm_slots = nn.LayerNorm(dim_slot)
        self.norm_mlp = nn.LayerNorm(dim_slot)

        self.project_q = nn.Linear(dim_slot, dim_slot, bias=False)
        self.project_k = nn.Linear(dim_in, dim_slot, bias=False)
        self.project_v = nn.Linear(dim_in, dim_slot, bias=False)

        self.gru = nn.GRUCell(dim_slot, dim_slot)
        self.mlp = nn.Sequential(
            nn.Linear(dim_slot, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, dim_slot),
        )

        # learned Gaussian init for slots
        self.slots_mu = nn.Parameter(torch.zeros(1, 1, dim_slot))
        self.slots_logsigma = nn.Parameter(torch.zeros(1, 1, dim_slot))

    def forward(
        self,
        x: torch.Tensor,
        slots_init: torch.Tensor | None = None,
        noise_std: float = 0.0,
    ) -> torch.Tensor:
        """
        x: (B, N, D_in)
        slots_init: (B, K, D_slot) or None
        noise_std: 初期スロットに足す微小ノイズ（0.0なら無し）
        """
        B, N, _ = x.shape
        x = self.norm_in(x)

        k = self.project_k(x)  # (B,N,D)
        v = self.project_v(x)  # (B,N,D)

        # init slots
        if slots_init is None:
            mu = self.slots_mu.expand(B, self.K, -1)
            sigma = self.slots_logsigma.exp().expand(B, self.K, -1)
            slots = mu + sigma * torch.randn_like(mu)
        else:
            # 受け取ったスロットを初期値に使う（同一性を維持）
            if slots_init.shape != (B, self.K, self.dim_slot):
                raise ValueError(f"slots_init shape mismatch: got {slots_init.shape}, expect {(B, self.K, self.dim_slot)}")
            slots = slots_init
            if noise_std > 0:
                slots = slots + noise_std * torch.randn_like(slots)

        for _ in range(self.iters):
            slots_prev = slots
            slots_norm = self.norm_slots(slots)
            q = self.project_q(slots_norm)  # (B,K,D)

            # attn logits: (B, N, K)
            attn_logits = torch.einsum("bnd,bkd->bnk", k, q) / math.sqrt(self.dim_slot)
            attn = F.softmax(attn_logits, dim=-1)  # over K

            # normalize over inputs N for each slot
            attn = attn + self.eps
            attn = attn / attn.sum(dim=1, keepdim=True)  # (B,N,K)

            updates = torch.einsum("bnk,bnd->bkd", attn, v)  # (B,K,D)

            # GRU
            slots = self.gru(
                updates.reshape(B * self.K, -1),
                slots_prev.reshape(B * self.K, -1),
            ).reshape(B, self.K, -1)

            slots = slots + self.mlp(self.norm_mlp(slots))

        return slots  # (B,K,D_slot)

# ---------------------------
# Cross-attention Decoder (grid queries attend to slots)
# ---------------------------
class GridSlotDecoder(nn.Module):
    """
    Given slots for one time step (B,K,D), produce logits over vocab for each grid position (H*W).
    """
    def __init__(self, dim_slot: int, dim_q: int, vocab_size: int, H: int = 32, W: int = 32):
        super().__init__()
        self.H, self.W = H, W
        self.N = H * W
        self.vocab_size = vocab_size

        self.pos = SinCosPosEmb2D(dim_q, H, W)  # (N, dim_q) これは固定でOK
        self.q_proj = nn.Linear(dim_q, dim_q, bias=False)
        self.k_proj = nn.Linear(dim_slot, dim_q, bias=False)
        self.v_proj = nn.Linear(dim_slot, dim_q, bias=False)

        self.out = nn.Sequential(
            nn.LayerNorm(dim_q),
            nn.Linear(dim_q, vocab_size),
        )

    def forward(self, slots: torch.Tensor) -> torch.Tensor:
        """
        slots: (B,K,D_slot)
        returns logits: (B, N, vocab)
        """
        B, K, _ = slots.shape

        # ★posはbufferなので、ここでdevice/dtypeへ合わせるだけ（軽い）
        # ★q_projは学習パラメータなので毎回通す（キャッシュしない！）
        pos = self.pos().to(device=slots.device, dtype=slots.dtype)    # (N, C)
        q = self.q_proj(pos).unsqueeze(0)                              # (1, N, C)  expandしない

        k = self.k_proj(slots)  # (B, K, C)
        v = self.v_proj(slots)  # (B, K, C)

        attn_logits = torch.einsum("bnc,bkc->bnk", q, k) / math.sqrt(q.shape[-1])  # broadcast OK
        attn = F.softmax(attn_logits, dim=-1)
        ctx = torch.einsum("bnk,bkc->bnc", attn, v)  # (B, N, C)

        logits = self.out(ctx)  # (B, N, vocab)
        return logits




# ---------------------------
# State Encoder: (34,25) -> (6, d_state)
#  - past17 -> 3
#  - fut17  -> 3
# ---------------------------
class StateEncoder(nn.Module):
    def __init__(self, state_dim: int = 25, d_state: int = 128, dropout: float = 0.0):
        super().__init__()
        self.state_dim = state_dim
        self.d_state = d_state

        # encode per raw-frame
        self.frame_mlp = nn.Sequential(
            nn.LayerNorm(state_dim),
            nn.Linear(state_dim, d_state),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(d_state, d_state),
        )

        # compress 17 -> 3 using 1D conv with stride-ish (simple & stable)
        # We'll do: reshape to (B, d_state, 17) and use conv to 3 timesteps.
        self.conv17_to_3 = nn.Conv1d(d_state, d_state, kernel_size=5, stride=6, padding=2)  # 17 -> 3 approx

        # small post
        self.post = nn.Sequential(
            nn.LayerNorm(d_state),
            nn.Linear(d_state, d_state),
            nn.ReLU(inplace=True),
        )

    def forward(self, states_34: torch.Tensor) -> torch.Tensor:
        """
        states_34: (B, 34, 25) float
        returns:   (B, 6, d_state)
        """
        B, T, D = states_34.shape
        assert T == 34 and D == self.state_dim

        x = self.frame_mlp(states_34)  # (B,34,d_state)
        past = x[:, :17]               # (B,17,d)
        fut  = x[:, 17:]               # (B,17,d)

        def compress17(seq17):
            # (B,17,d) -> (B,d,17)
            z = seq17.transpose(1, 2)
            z = self.conv17_to_3(z)    # (B,d,3) because 17 -> 3 with stride 6
            z = z.transpose(1, 2)      # (B,3,d)
            return z

        past3 = compress17(past)
        fut3  = compress17(fut)
        out = torch.cat([past3, fut3], dim=1)  # (B,6,d_state)
        out = self.post(out)
        return out


# ---------------------------
# Token Encoder: tokens (B,3,32,32) -> features (B,3,N,d_in)
# ---------------------------
class TokenEncoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 256, H: int = 32, W: int = 32, dropout: float = 0.0):
        super().__init__()
        self.H, self.W = H, W
        self.N = H * W
        self.d = d_model

        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos2d = SinCosPosEmb2D(d_model, H, W)  # (N,d)
        self.mlp = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tok_3hw: torch.Tensor) -> torch.Tensor:
        """
        tok_3hw: (B, 3, H, W) int64
        returns: feats (B, 3, N, d)
        """
        B, T, H, W = tok_3hw.shape
        assert T == 3 and H == self.H and W == self.W

        x = self.embed(tok_3hw)  # (B,3,H,W,d)
        x = x.view(B, T, self.N, self.d)  # (B,3,N,d)

        pos = self.pos2d().unsqueeze(0).unsqueeze(0)  # (1,1,N,d)
        x = x + pos
        x = self.mlp(x)
        return x  # (B,3,N,d)


# ---------------------------
# Slot Dynamics: predict future slots using state sequence
# ---------------------------
class SlotDynamicsTransformer(nn.Module):
    """
    Update slots along time using a small Transformer encoder.
    - We build 6 time steps: past(0..2) observed slots, future(3..5) are learned "mask slots".
    - Condition on state6 by adding projected state embedding at each time step.
    """
    def __init__(
        self,
        dim_slot: int,
        d_state: int,
        n_layers: int = 2,
        n_heads: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.D = dim_slot

        self.state_proj = nn.Linear(d_state, dim_slot)

        # learned mask token for future slots (one per slot position)
        self.future_slot_token = nn.Parameter(torch.zeros(1, 1, 1, dim_slot))  # (1,1,1,D)

        # time positional embedding (6 steps)
        self.time_pos = nn.Parameter(torch.zeros(1, 6, 1, dim_slot))  # (1,6,1,D)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=dim_slot,
            nhead=n_heads,
            dim_feedforward=4 * dim_slot,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(dim_slot)

        nn.init.normal_(self.future_slot_token, std=0.02)
        nn.init.normal_(self.time_pos, std=0.02)

    def forward(self, slots_past: torch.Tensor, state_seq6: torch.Tensor) -> torch.Tensor:
        """
        slots_past: (B,3,K,D)
        state_seq6: (B,6,d_state)
        returns:    slots_future: (B,3,K,D) for t=3,4,5
        """
        B, Tp, K, D = slots_past.shape
        assert Tp == 3 and D == self.D
        assert state_seq6.shape[1] == 6

        # Build 6-step slots: past known + future masked
        future = self.future_slot_token.expand(B, 3, K, D)  # (B,3,K,D)
        slots6 = torch.cat([slots_past, future], dim=1)     # (B,6,K,D)

        # Add conditioning: state + time pos
        st = self.state_proj(state_seq6).unsqueeze(2)       # (B,6,1,D)
        x = slots6 + st + self.time_pos                    # (B,6,K,D)

        # Time-only Transformer: treat each slot independently over time
        x = x.permute(0, 2, 1, 3).reshape(B * K, 6, D)      # (B*K,6,D)
        x = self.encoder(x)                                 # (B*K,6,D)
        x = self.norm(x)
        x = x.reshape(B, K, 6, D).permute(0, 2, 1, 3)       # (B,6,K,D)

        return x[:, 3:, :, :]  # (B,3,K,D)

# ---------------------------
# Full Model
# ---------------------------
@dataclass
class ModelConfig:
    vocab_size: int
    num_slots: int = 8
    d_token: int = 256
    d_slot: int = 256
    d_state: int = 128
    slot_iters: int = 3
    H: int = 32
    W: int = 32


class SlotPredictor(nn.Module):
    """
    Input:
      - past_tokens: (B,3,32,32) int64
      - robot_states: (B,34,25) float32
    Output:
      - logits_future: (B,3,HW,V)
    """
    def __init__(self, cfg: "ModelConfig"):
        super().__init__()
        self.cfg = cfg

        self.token_enc = TokenEncoder(cfg.vocab_size, cfg.d_token, cfg.H, cfg.W)
        self.slot_attn = SlotAttention(
            num_slots=cfg.num_slots,
            dim_in=cfg.d_token,
            dim_slot=cfg.d_slot,
            iters=cfg.slot_iters,
            hidden_dim=max(128, cfg.d_slot),
        )

        self.state_enc = StateEncoder(state_dim=25, d_state=cfg.d_state)

        self.in_proj = nn.Linear(cfg.d_token, cfg.d_token)

        self.dynamics = SlotDynamicsTransformer(
            dim_slot=cfg.d_slot,
            d_state=cfg.d_state,
            n_layers=2,
            n_heads=4,
            dropout=0.1,
        )

        self.decoder = GridSlotDecoder(
            dim_slot=cfg.d_slot,
            dim_q=cfg.d_slot,
            vocab_size=cfg.vocab_size,
            H=cfg.H, W=cfg.W
        )

    def forward(
        self,
        past_tokens_3hw: torch.Tensor,
        robot_states_34x25: torch.Tensor,
        *,
        detach_slot_carry: bool = True,
        slot_init_noise_std: float = 0.005,
    ) -> torch.Tensor:
        """
        past_tokens_3hw: (B,3,32,32) long
        robot_states_34x25: (B,34,25) float
        detach_slot_carry:
          - True: t=0→1→2 の carry-overを detach（推奨：軽い/安定）
          - False: 勾配も流す（重い/不安定になり得る）
        slot_init_noise_std:
          - carry-over時に初期スロットへ加えるノイズ（0.0〜0.01目安）
        returns logits: (B,3,N,V)
        """
        B, T, H, W = past_tokens_3hw.shape
        assert T == 3 and H == self.cfg.H and W == self.cfg.W
        N = H * W

        feats = self.token_enc(past_tokens_3hw)  # (B,3,N,d_token)

        # ---- Slot carry-over over time ----
        slots_past = []
        slots_init = None  # (B,K,D) or None
        for t in range(3):
            x_t = self.in_proj(feats[:, t])  # (B,N,d_token)

            s_t = self.slot_attn(
                x_t,
                slots_init=slots_init,
                noise_std=(slot_init_noise_std if slots_init is not None else 0.0),
            )  # (B,K,d_slot)

            slots_past.append(s_t)

            # carry to next step
            slots_init = s_t.detach() if detach_slot_carry else s_t

        slots_past = torch.stack(slots_past, dim=1)  # (B,3,K,d_slot)

        state6 = self.state_enc(robot_states_34x25)   # (B,6,d_state)
        slots_future = self.dynamics(slots_past, state6)  # (B,3,K,d_slot)

        # decode each future time step to vocab logits over grid
        logits = torch.stack(
            [self.decoder(slots_future[:, t]) for t in range(3)],
            dim=1
        )  # (B,3,N,V)

        return logits

In [3]:
from torch.utils.data import DataLoader
import wandb

def collate_fn(batch):
    # batch: list of dicts
    past = torch.stack([torch.from_numpy(b["past_frames"]) for b in batch], dim=0).long()      # (B,3,32,32)
    fut  = torch.stack([torch.from_numpy(b["future_frames"]) for b in batch], dim=0).long()    # (B,3,32,32)
    st   = torch.stack([torch.from_numpy(b["robot_states"]) for b in batch], dim=0).float()    # (B,34,25)
    return {"past": past, "future": fut, "states": st}

import time
import torch
import torch.nn.functional as F
from torch import nn

def train_one_epoch_wandb_amp(
    model,
    loader,
    optimizer,
    scheduler,
    device="cuda",
    grad_clip=1.0,
    log_every=50,
    epoch=0,
    global_step=0,
    amp_dtype="bf16",   # "bf16" or "fp16"
    scaler=None,        # 外から渡す（推奨）
):
    assert device.startswith("cuda"), "AMPはCUDA前提"
    model.train()

    # GradScaler: fp16では必須、bf16では基本不要だが、統一のため使ってOK
    if scaler is None:
        scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == "fp16"))

    # autocast dtype
    autocast_dtype = torch.bfloat16 if amp_dtype == "bf16" else torch.float16
    autocast_enabled = True

    total_loss = 0.0
    total_tokens = 0
    ema = None
    ema_beta = 0.98
    t0 = time.time()

    pbar = tqdm(loader, desc=f"train epoch {epoch}", leave=False)

    for step, batch in enumerate(pbar):
        past = batch["past"].to(device, non_blocking=True)
        future = batch["future"].to(device, non_blocking=True)
        states = batch["states"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # ★ここがAMPを挟む場所（forward〜loss）
        with torch.cuda.amp.autocast(enabled=autocast_enabled, dtype=autocast_dtype):
            logits = model(past, states)  # (B,3,N,V)
            B, T, N, V = logits.shape
            target = future.view(B, T, -1)  # (B,3,1024)

            loss = F.cross_entropy(
                logits.reshape(B * T * N, V),
                target.reshape(B * T * N),
                reduction="mean",
            )

        # ★backward/stepはscaler経由（fp16のとき安全）
        if amp_dtype == "fp16":
            scaler.scale(loss).backward()
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            # bf16ならscaler無しでOK（多くの環境で高速・安定）
            loss.backward()
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        
        scheduler.step()

        tokens = B * T * N
        loss_val = float(loss.item())
        total_loss += loss_val * tokens
        total_tokens += tokens
        ema = loss_val if ema is None else (ema_beta * ema + (1 - ema_beta) * loss_val)

        dt = time.time() - t0
        tok_per_s = total_tokens / max(dt, 1e-9)
        lr = optimizer.param_groups[0]["lr"]

        wandb.log(
            {
                "train/loss_step": loss_val,
                "train/loss_ema": float(ema),
                "train/lr": lr,
                "train/tokens_seen": total_tokens,
                "train/tok_per_sec": tok_per_s,
                "epoch": epoch,
                "amp/dtype": amp_dtype,
            },
            step=global_step,
        )

        pbar.set_postfix(
            loss=f"{loss_val:.4f}",
            ema=f"{ema:.4f}",
            lr=f"{lr:.2e}",
            tok_s=f"{tok_per_s:,.0f}",
        )

        global_step += 1

    epoch_loss = total_loss / max(1, total_tokens)
    wandb.log({"train/loss_epoch": epoch_loss}, step=global_step)
    return epoch_loss, global_step, scaler

In [4]:
device = "cuda"
root = "/root/work/data/raw/train_v2.0"

from dataclasses import asdict
from torch.optim.lr_scheduler import LambdaLR
# ---- single source of truth ----
cfg = ModelConfig(
    vocab_size=65536,
    num_slots=6,
    d_token=128,
    d_slot=128,
    d_state=128,
    slot_iters=3,
    H=32, W=32
)

train_cfg = {
    "batch_size": 8,
    "lr": 8e-4,
    "weight_decay": 1e-5,
    "grad_clip": 5.0,
    "log_every": 10,
    "epochs": 1,
    "num_workers": 4,
    "warmup_steps": 200,
}


wandb.init(
    project="slot-attn-token-pred",
    name="run4",
    config={**asdict(cfg), **train_cfg},  # cfg をそのまま wandb に同期
)


model = SlotPredictor(cfg).to(device)

ds = TrainDataset(
    root="/root/work/data/raw/train_v2.0",
    output_format="seq2seq",
    cache_path="/root/work/data/outputs/valid_starts_stride3_clipclean.npy",
)

loader = DataLoader(
    ds,
    batch_size=train_cfg["batch_size"],
    shuffle=True,
    num_workers=train_cfg["num_workers"],
    pin_memory=True,
    collate_fn=collate_fn,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_cfg["lr"],
    weight_decay=train_cfg["weight_decay"],
)
base_lr = optimizer.param_groups[0]["lr"]
epochs = train_cfg["epochs"]
steps_per_epoch = len(loader)
total_steps = epochs * steps_per_epoch
warmup_steps = train_cfg["warmup_steps"]
print("steps_per_epoch:", steps_per_epoch, "total_steps:", total_steps, "warmup_steps:", warmup_steps)
def lr_lambda(step: int):
    # step: 0,1,2,... (global step)
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    # cosine decay: warmup後は 1 -> 0 へ
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = None
global_step = 0
for epoch in range(train_cfg["epochs"]):
    avg, global_step, scaler = train_one_epoch_wandb_amp(
        model, loader, optimizer, scheduler,
        device="cuda",
        grad_clip=train_cfg["grad_clip"],
        log_every=train_cfg["log_every"],
        epoch=epoch,
        global_step=global_step,
        amp_dtype="bf16",  # bf16が使えるGPUならこれ
        scaler=scaler
    )

wandb.finish()


wandb: Currently logged in as: 1kbooks-equity (1kbooks-equity-the-university-of-tokyo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/root/work/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/tmp/ipykernel_92725/3376902762.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == "fp16"))


steps_per_epoch: 77454 total_steps: 77454 warmup_steps: 200


train epoch 0:   0%|          | 0/77454 [00:00<?, ?it/s]/tmp/ipykernel_92725/3376902762.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=autocast_enabled, dtype=autocast_dtype):


KeyboardInterrupt: 

In [5]:
import torch
from torch.profiler import profile, ProfilerActivity

def profile_n_steps(loader, model, optimizer, device="cuda", n=10):
    model.train()
    it = iter(loader)

    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        for i in range(n):
            batch = next(it)
            past   = batch["past"].to(device, non_blocking=True)
            future = batch["future"].to(device, non_blocking=True)
            states = batch["states"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(past, states)  # (B,3,N,V)
            B, T, N, V = logits.shape
            target = future.view(B, T, -1)

            loss = torch.nn.functional.cross_entropy(
                logits.reshape(B*T*N, V),
                target.reshape(B*T*N),
                reduction="mean",
            )

            loss.backward()
            optimizer.step()

            # ★グラフが残らないように明示的に切る（念のため）
            del logits, loss, past, future, states, batch
            prof.step()

    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=30))


# 使い方
profile_n_steps(loader, model, optimizer, device="cuda", n=10)


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
    autograd::engine::evaluate_function: AddmmBackward0         0.01%       4.029ms         5.70%        3.954s      13.182ms       0.000us         0.00%       26.739s      89.130ms           0 B           0 B     -59.79 GB     -60.48 G